# Template Definition

In [1]:
ROOT_FOLDER = "../dataset"

In [2]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
prompt_template = PromptTemplate.from_template(
"""
You will be asked by the user to create a textual description from a PlantUML description. To do so in the most clear way possible, avoid class properties. 
If there are any models that have a cardinality of nothing, assume that it would be "0 to many".
If a cardinality has a specific number above one, make that cardinality into "many" and keep the lower multiplicity. If it is "0 to 4", it would become "0 to many". If it is "1 to 20", it would become "1 to many".
If only a "*" appears, treat it as 0 to many. If only * appears, treat it as 1 to many.
If an association class appears, it will be denoted as .. link between two pairs of classes. For example, Article .. Author and Article .. Person. Transform it into, Author is an association class to association Article and Person.
Include abstract classes and enumerations that are connected to other classes as regular classes. For example, Employee <|-- OtherEmployee, would become, OtherEmployee has 0 to many employee and employee has 0 to many OtherEmployee.
Do not include note and anything that the not is identified as. For example, note "{{XOR}}" as N1and Book .. N1, as note is identified as N1, do not include these lines in the generation process.
Do not include enumerations that are connected to nothing.
Do not include association descriptions. For example, Planner "1" --"*" Order : creates, would become, Order has exactly 1 Planner and Planner has 0 to many order.
Do not include association direction. For example, Employee "1" -- "0..1" Faculty : leads >, would become Employee has 0 to 1 Faculty and Faculty has exactly 1 Employee.
Output only the textual description and nothing else.

###############

Here is an example of how it should be done.

The PlantUML description is:
@startuml

enum Position <<enum>> {{
    Goalkeeper
    LeftBack
}}

enum Recommendation <<enum>> {{
    KeyPlayer
    FirstTeamPlayer
    ReserveTeamPlayer
    ProspectivePlayer
    NotRecommended
}}

enum ScoutingStatus <<enum>> {{
    LongListed
    ShortListed
    RecommendedForSigning
    OfferMade
}}

enum ScoutKind <<enum>> {{
    RegularScout
    HeadScout
}}

abstract class Person {{
    String FirstName
    String LastName
}}

class Player {{
    ScoutingStatus Status
}}

class HeadCoach {{}}

class Director {{}}

class Scout {{
    ScoutKind ScoutKind
}}

class Club {{}}

class Offer {{
    Int Value
}}

class ScoutingAssignment {{}}

class ScoutReport {{
    String Pro
    String Con
    Recommendation Recommendation
}}

class PlayerProfile {{
    Position Position
}}

class PlayerAttribute {{
    String Name
    Int Value
}}

Person <|-- Player
Person <|-- HeadCoach
Person <|-- Director
Person <|-- Scout

Director *--> "0..*" Offer
Club *--> "0..1" Director
Club *--> "0..1" HeadCoach
Club *--> "0..*" Player
Club *--> "0..*" Scout
Scout *--> "0..*" ScoutingAssignment
ScoutingAssignment *--> "0..1" ScoutReport 
HeadCoach *--> "0..*" PlayerProfile
PlayerProfile *--> "0..*" PlayerAttribute


Player "0..1" <--> "0..*" Offer
Player "0..1" <--> "0..1" PlayerProfile
Player "1..1" <--> "0..*" ScoutingAssignment
ScoutReport "0..1" <--> "0..1" ScoutReport : nextReport

@enduml

This model would result in a description like:
Club has 0 to 1 HeadCoach and HeadCoach has 1 to many Club. Head Coach has 0 to many PlayerProfile and PlayerProfile has 1 to many HeadCoach. PlayerProfile has 0 to many PlayerAttribute and PlayerAttribute has 1 to many PlayerProfile. Club has 0 to many Player and Player has 1 to many Club. Player has 0 to 1 PlayerProfile and PlayerProfile has 0 to 1 Player. Player has 0 to many ScoutingAssignment and ScoutingAssignment has exactly 1 Player. ScoutingAssignment has 0 to 1 ScoutReport and ScoutReport has 1 to many ScoutingAssignment. ScoutReport has 0 to 1 ScoutReport and ScoutReport has 0 to 1 ScoutReport. Player has 0 to many Offer and Offer has 0 to 1 Player. Club has 0 to many Scout and Scout has 1 to many Club. Scout has 0 to many ScoutingAssignment and ScoutingAssignment has 1 to many Scout. Club has 0 to 1 Director and Director has 1 to many Club. Director has 0 to many Offer and Offer has 1 to many Director. Person has 0 to many HeadCoach and HeadCoach has 0 to many Person. Person has 0 to many Player and Player has 0 to many Person. Person has 0 to many Scout and Scout has 0 to many Person. Person has 0 to many Director and Director has 0 to many Person.

Another example is this.

The PlantUML description is:
@startuml

class Game {{
  int ID
}}

class Player {{
  int ID
}}

class Deck {{
  int ID
}}

class Card {{
  int ID
  Suit suit
  Compare compare(Card)
}}

class Scoreboard {{
  int ID
}}

class Round {{
  int ID
  int points
}}

abstract class RoundType {{
  int ID
}}

class SoloRound {{
  SoloType solo
}}

class MandatorySolo

class DoubleRound 

enum Suit <<enum>> {{
  Diamonds
  Hearts
  Spades
  Clubs
}}

enum Compare <<enum>> {{
  Lower
  Higher
  Equal
}}

enum SoloType <<enum>> {{
  Trump
  Jacks
  Queens
}}

Game --> "1" Scoreboard
Game --> "1" Deck
Game --> "4" Player

Deck --> "0..48" Card

Scoreboard --> "*" Round

Player --> "1..3" Round
Player --> "1..2" RoundType

RoundType <|-- SoloRound
RoundType <|-- DoubleRound
SoloRound <|-- MandatorySolo

Card --> Suit
Card --> Compare
SoloRound --> SoloType
@enduml

This model would result in a description like:
Game has exactly 1 Deck and Deck has 0 to many Game. Deck has 0 to many Card and Card has 0 to many Deck. Card has 0 to many Suit and Suit has 0 to many Card. Card has 0 to many Compare and Compare has 0 to many Card. Game has exactly 1 Scoreboard and Scoreboard has 0 to many Game. Scoreboard has 1 to many Round and Round has 0 to many Scoreboard. Game has 1 to many Player and Player has 0 to many Game. Player has 1 to many Round and Round has 0 to many Player. Player has 1 to many RoundType and RoundType has 0 to many Player. RoundType has 0 to many DoubleRound and DoubleROund has 0 to many RoundType. RoundType has 0 to many SoloRound and SoloRound has 0 to many RoundType. SoloRound has 0 to many SoloType and SoloType has 0 to many SoloRound. SoloRound has 0 to many MandatorySolo and MandatorySolo has 0 to many SoloRound.

###############

The model to make into a description is:

{text}

###############

Based on the examples, the model description is:

"""
)

In [3]:
import sys
import threading
from time import sleep
try:
    import thread
except ImportError:
    import _thread as thread

def quit_function(fn_name):
    # print to stderr, unbuffered in Python 2.
    print('{0} took too long'.format(fn_name), file=sys.stderr)
    sys.stderr.flush() # Python 3 stderr is likely buffered.
    thread.interrupt_main() # raises KeyboardInterrupt
    
def exit_after(s):
    '''
    use as decorator to exit process if 
    function takes longer than s seconds
    '''
    def outer(fn):
        def inner(*args, **kwargs):
            timer = threading.Timer(s, quit_function, args=[fn.__name__])
            timer.start()
            try:
                result = fn(*args, **kwargs)
            finally:
                timer.cancel()
            return result
        return inner
    return outer

In [4]:
import os
from tqdm import tqdm
from docx import Document

from langchain_core.tracers.context import tracing_v2_enabled

@exit_after(360)
def run_chain(chain, text):
    return chain.invoke({"text": text})

def process_subfolders_with_chain(root_folder_path, chain, type=''):
    """
    Explores subfolders of the root folder (depth 1), processes each subfolder's `text.txt`
    with the provided LangChain chain, and saves the result in a new file in the same folder.

    Args:
        root_folder_path (str): Path to the root folder.
        chain: A LangChain chain instance to process text inputs.
    """
    for subfolder_name in tqdm(os.listdir(root_folder_path)):
        subfolder_path = os.path.join(root_folder_path, subfolder_name)
        
        # Ensure the current item is a subfolder
        if os.path.isdir(subfolder_path):
            text_file_path = os.path.join(subfolder_path, "plantuml.txt")

            # Recheck if `text.txt` now exists
            if os.path.isfile(text_file_path):
                with open(text_file_path, "r", encoding="utf-8") as file:
                    text = file.read()

                with tracing_v2_enabled():
                    # Call the LangChain chain with the input dictionary
                    try:
                        result = run_chain(chain, text)
                    except:
                        result = ""

                # Save the result to a new file in the same subfolder
                result_file_path = os.path.join(subfolder_path, f"{type}_literal_description_generated_from_original_plantuml.txt")
                with open(result_file_path, "w", encoding="utf-8") as result_file:
                    result_file.write(result)

In [5]:
def delete_result_txt_files(root_folder_path):
    """
    Deletes every .txt file that starts with 'result_' in the subfolders of the root folder (depth 1).

    Args:
        root_folder_path (str): Path to the root folder.
    """
    for subfolder_name in os.listdir(root_folder_path):
        subfolder_path = os.path.join(root_folder_path, subfolder_name)
        
        # Ensure the current item is a subfolder
        if os.path.isdir(subfolder_path):
            for file_name in os.listdir(subfolder_path):
                if file_name.startswith("result_") and file_name.endswith(".txt"):
                    file_path = os.path.join(subfolder_path, file_name)
                    os.remove(file_path)

In [6]:
#delete_result_txt_files(ROOT_FOLDER)

# Two Shot Open-AI

In [7]:
from dotenv import load_dotenv
assert load_dotenv()

In [8]:
MODEL_OPEN_AI = ["gpt-5-mini","gpt-5.2","gpt-5-nano", "o3-mini", "gpt-4o-mini", "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo"]

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display_markdown

In [10]:
def open_ai_two(model):
    model_ = ChatOpenAI(model=model, reasoning_effort="low")
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [11]:
def open_ai_make_example(model):
    model = ChatOpenAI(model=model)
    chain = prompt_template | model | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res

In [12]:
display_markdown(open_ai_make_example(MODEL_OPEN_AI[0]), raw=True)

Customer has exactly 1 Broker and Broker has 0 to many Customer. Customer has 0 to many InsuranceProduct and InsuranceProduct has 0 to many Customer. Broker has 0 to many CustomerProfile and CustomerProfile has 0 to many Broker. HeadOffice has 0 to many CustomerProfile and CustomerProfile has 0 to many HeadOffice. Customer has 0 to many Offer and Offer has exactly 1 Customer. Contract has exactly 1 Offer and Offer has 0 to many Contract. Contract has exactly 1 InsuranceProduct and InsuranceProduct has 0 to many Contract. Contract has 0 to many Invoice and Invoice has exactly 1 Contract. Invoice has exactly 1 Customer and Customer has 0 to many Invoice. Customer has 0 to many Claim and Claim has exactly 1 Customer. Claim has 1 to many ClaimCase and ClaimCase has exactly 1 Claim. ClaimCase has 0 to many Estimator and Estimator has 0 to many ClaimCase. Estimator has 0 to many EstimatorReport and EstimatorReport has exactly 1 Estimator. ClaimCase has 0 to many EstimatorReport and EstimatorReport has exactly 1 ClaimCase. ClaimCase has 0 to many Document and Document has exactly 1 ClaimCase. ClaimCase has 0 to many CompensationDecision and CompensationDecision has exactly 1 ClaimCase. CompensationDecision has exactly 1 Payment and Payment has exactly 1 CompensationDecision. Payment has exactly 1 Account and Account has 0 to many Payment. Customer has 0 to many Account and Account has exactly 1 Customer. Payment has 0 to many EstimatorReport and EstimatorReport has 0 to many Payment.

In [13]:
for model in MODEL_OPEN_AI[1:2]:
    print(f"Few shot with {model}")
    open_ai_two(model)

Few shot with gpt-5.2


100%|██████████| 48/48 [10:31<00:00, 13.16s/it]
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## Two Shot Anthropic

In [14]:
from langchain_anthropic import ChatAnthropic

In [15]:
MODEL_ANTHROPIC = ["claude-sonnet-4-5-20250929","claude-3-7-sonnet-20250219"]

In [16]:
def anthropic_make_example(model):
    model = ChatAnthropic(model=model,temperature=0,
    max_tokens=4096,
    timeout=None,
    max_retries=2,)
    chain = prompt_template | model | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res

In [17]:
def anthropic_two(model):
    model_ = ChatAnthropic(model=model)
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [18]:
display_markdown(anthropic_make_example(MODEL_ANTHROPIC[0]), raw=True)

@startuml

class Customer {}
class Broker {}
class BrokerCustomerAssignment {}
class InsurancePolicy {}
class Contract {}
class Invoice {}
class ClaimCase {}
class Estimator {}
class Report {}
class CompensationPayment {}

Customer "1" -- "0..1" BrokerCustomerAssignment
Broker "1" -- "0..*" BrokerCustomerAssignment
Customer "1" -- "0..*" Contract
Contract "1" -- "0..*" InsurancePolicy
Contract "1" -- "0..*" Invoice
Customer "1" -- "0..*" ClaimCase
ClaimCase "1" -- "0..*" Report
Report "0..*" -- "1" Estimator
ClaimCase "1" -- "0..*" CompensationPayment

@enduml

In [19]:
for model in MODEL_ANTHROPIC:
    print(f"Two shot with {model}")
    anthropic_two(model)

Two shot with claude-sonnet-4-5-20250929


100%|██████████| 48/48 [03:38<00:00,  4.56s/it]


Two shot with claude-3-7-sonnet-20250219


100%|██████████| 48/48 [03:55<00:00,  4.90s/it]


# Two Shot Open LLM

In [20]:
import torch, gc

## Deepseek

In [21]:
from langchain_deepseek import ChatDeepSeek

In [22]:
MODEL_DEEPSEEK = ["deepseek-chat"] #, "deepseek-reasoner"]

In [23]:
def deepseek_make_example(model):
    model = llm = ChatDeepSeek(
            model=model,
            temperature=0,
            max_tokens=None,
            timeout=None,
            max_retries=2,
            # api_key="...",
            # other params...
        )
    chain = prompt_template | model | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res

In [24]:
def deepseek_two(model):
    model_ = ChatDeepSeek(model=model)
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [25]:
%%time
display_markdown(deepseek_make_example(MODEL_DEEPSEEK[0]), raw=True)

@startuml

class Customer {}
class InsurancePolicy {}
class Contract {}
class Invoice {}
class Broker {}
class ClaimCase {}
class Report {}
class Estimator {}
class CompensationPayment {}
class BrokerAssignment {}

Customer "1" -- "0..*" Contract
Contract "1" -- "0..*" InsurancePolicy
Contract "1" -- "0..*" Invoice
Contract "1" -- "0..*" ClaimCase
Customer "1" -- "0..*" BrokerAssignment
Broker "1" -- "0..*" BrokerAssignment
ClaimCase "1" -- "0..*" CompensationPayment
ClaimCase "1" -- "0..*" Report
Report "0..*" -- "1" Estimator

@enduml

CPU times: user 64.9 ms, sys: 11.8 ms, total: 76.7 ms
Wall time: 7.1 s


In [26]:
for model in MODEL_DEEPSEEK:
    print(f"Two shot with {model}")
    deepseek_two(model)

Two shot with deepseek-chat


100%|██████████| 48/48 [05:06<00:00,  6.38s/it]


## Ollama

In [27]:
from langchain_ollama import ChatOllama

In [28]:
MODEL_OLLAMA = [] #["llama3.2:3b-text-fp16"]

In [29]:
def ollama_two(model):
    model_ = ChatOllama(
        model=model,
        temperature=0,
        timeout = 3,
    )
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [30]:
def ollama_make_example(model):
    model = ChatOllama(
        model=model,
        temperature=0,
        timeout = 3,
    )
    chain = prompt_template | model | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res
    

In [31]:
#display_markdown(ollama_make_example(MODEL_OLLAMA[0]), raw=True)

In [32]:
for model in MODEL_OLLAMA:
    print(f"Few shot with {model}")
    ollama_zero(model)
    gc.collect()
    torch.mps.empty_cache()

# Mistral

In [33]:
from langchain_mistralai import ChatMistralAI

In [34]:
MODEL_MISTRAL = ["mistral-large-2411", "codestral-2508", "mistral-small-2506"]

In [35]:
def mistral_few(model):
    model_ = ChatMistralAI(model=model)
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [36]:
def mistral_make_example(model):
    model = ChatMistralAI(model=model)
    chain = prompt_template | model | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system\so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res

In [37]:
display_markdown(mistral_make_example(MODEL_MISTRAL[0]), raw=True)

```plantuml
@startuml

class Customer {}
class InsurancePolicy {}
class Contract {}
class Invoice {}
class Broker {}
class ClaimCase {}
class Report {}
class Estimator {}
class CompensationPayment {}

Customer "1" -- "0..*" Contract
Contract "1" -- "0..*" Invoice
Contract "0..*" -- "1" InsurancePolicy
Contract "1" -- "0..*" ClaimCase
Broker "1" -- "0..*" Customer
ClaimCase "1" -- "0..*" Report
Report "0..*" -- "1" Estimator
ClaimCase "1" -- "0..*" CompensationPayment

@enduml
```

In [38]:
for model in MODEL_MISTRAL:
    print(f"Few shot with {model}")
    mistral_few(model)

Few shot with mistral-large-2411


100%|██████████| 48/48 [02:55<00:00,  3.66s/it]


Few shot with codestral-2508


100%|██████████| 48/48 [00:40<00:00,  1.18it/s]


Few shot with mistral-small-2506


100%|██████████| 48/48 [02:05<00:00,  2.61s/it]


# Gemini

In [39]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [40]:
MODEL_GEMINI = ["gemini-2.5-flash"]

In [41]:
def gemini_few(model):
    model_ = ChatGoogleGenerativeAI(model=model)
    chain = prompt_template | model_ | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model)

In [42]:
def gemini_make_example(model):
    model = ChatGoogleGenerativeAI(model=model)
    chain = prompt_template | model | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system\so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res

In [43]:
display_markdown(gemini_make_example(MODEL_GEMINI[0]), raw=True)

@startuml

class Customer {}
class InsurancePolicy {}
class Contract {}
class Invoice {}
class BrokerCustomerAssignment {}
class Broker {}
class ClaimCase {}
class Report {}
class Estimator {}
class CompensationPayment {}

Contract -- "1" Customer
Contract "1" -- Invoice
Contract "1" -- InsurancePolicy
Contract "1" -- ClaimCase
ClaimCase "1" -- CompensationPayment
ClaimCase "1" -- Report
Report -- "1" Estimator
BrokerCustomerAssignment "0..1" -- "1" Customer
Broker "1" -- BrokerCustomerAssignment

@enduml

In [44]:
for model in MODEL_GEMINI:
    print(f"Few shot with {model}")
    gemini_few(model)

Few shot with gemini-2.5-flash


100%|██████████| 48/48 [28:25<00:00, 35.54s/it]


## Huggingface

In [45]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace, HuggingFaceEndpoint
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [46]:
MODEL_HUGGINGFACE = [] #["Qwen/Qwen2.5-3B-Instruct", "microsoft/Phi-3-mini-4k-instruct", "google/gemma-2-27b-it"]

In [47]:
def huggingface_two(model):
    model_id = model
    
    llm = HuggingFaceEndpoint(
        repo_id=model_id,
        task="text-generation",
        max_new_tokens=2048,
        do_sample=False,
        repetition_penalty=1.03,
        temperature=0.01,
    )

    chat = ChatHuggingFace(llm=llm, verbose=True)
    
    chain = prompt_template | chat | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model_id.replace("/","_"))

In [48]:
def huggingface_make_example(model):

    model_id = model
    
    

    llm = HuggingFacePipeline.from_model_id(
        model_id=model_id,
        task="text-generation",
        pipeline_kwargs={"temperature": 0.1, "max_new_tokens": 1024}
    )

    chat = ChatHuggingFace(llm=llm, verbose=True)
    
    chain = prompt_template | chat | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
        As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.
        
        In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 
        
        """})
    return res


In [49]:
#display_markdown(huggingface_make_example(MODEL_HUGGINGFACE[0]), raw=True)

In [50]:
for model in MODEL_HUGGINGFACE:
    print(f"Few shot with {model}")
    huggingface_two(model)
    gc.collect()
    torch.mps.empty_cache()

## Mlx-LLM

In [51]:
from langchain_community.llms import MLXPipeline
from langchain_community.chat_models import ChatMLX

In [52]:
MODEL_MLX = ["mlx-community/phi-4-8bit",
             "mlx-community/Falcon3-10B-Instruct-6bit", 
             "mlx-community/Qwen3-4B-Instruct-2507-4bit-DWQ-2510",
             "mlx-community/Mistral-7B-Instruct-v0.3-4bit",
             "mlx-community/Llama-3.2-3B-Instruct-4bit",
             "mlx-community/gemma-3-4b-it-5bit",
             #"mlx-community/gemma-2-27b-it-4bit",
            # "mlx-community/Mamba-Codestral-7B-v0.1-8bit",
            #"mlx-community/CodeLlama-13b-Instruct-hf-4bit-MLX"
            ]

In [53]:
def mlx_two(model):
    llm = MLXPipeline.from_model_id(
        model_id=model,
        pipeline_kwargs={"max_tokens": 15_000, "temp": 0.1},
    )
    chat = ChatMLX(llm=llm)
    chain = prompt_template | chat | StrOutputParser()
    process_subfolders_with_chain(ROOT_FOLDER, chain, type=model.replace("/","_"))

In [54]:
def mlx_make_example(model):
    llm = MLXPipeline.from_model_id(
        model_id=model,
        pipeline_kwargs={"max_tokens": 2000, "temp": 0.7},
    )
    chat = ChatMLX(llm=llm)
    chain = prompt_template | chat | StrOutputParser()
    res = chain.invoke({"text": """Alpha Insurance is an insurance company that provides its clients with various types of insurance policies. 
As soon as a customer addresses Alpha Insurance, a broker is assigned to follow the customer’s file. The broker is registered in the system, so that when a customer calls, based on the contract, the help desk can immediately trace who is the customer's first account manager. After the broker is assigned to the customer, the latter indicates which type(s) of insurance policy they would like to sign for, so the broker could, depending on the case, either assess the customer’s profile on the spot or send the customer’s file for analysis to the head office. After the customer’s profile has been assessed and the customer has been deemed trustworthy , a preliminary contract/offer on an insurance product is made to the customer either in person or by email. (Such offers can also be extended to already existing customers.) If the customer agrees to the offer, the contract is signed by both parties. After the signing of the contract, the client enjoys the coverage and is invoiced (monthly or yearly – depending on the choice made in the contract) according to the price of the insurance product they bought.

In case the insured event happens, a customer should send a claim for compensation. Then the company opens one or several claim cases (e.g. in case of an accident, often material damage & physical damage are handled separately). Once the case file is complete, it is sent for assessment by different estimators based on their area of expertise. According to the reports issued by the estimators, it is decided whether the claim case is approved. In case of approval the compensation decision is registered that stipulates which costs are eligible for (partial) refund.  For the supplied documents, the sum of compensation is calculated and the compensation is paid to the customer’s account.  The estimators’ reports must be stored in the database for at least one year after the payment of compensation for legal purposes. 

"""})
    return res


In [55]:
display_markdown(mlx_make_example(MODEL_MLX[2]), raw=True)

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

@startuml

class Customer {}
class InsurancePolicy {}
class Contract {}
class Invoice {}
class Broker {}
class BrokerCustomerAssignment {}
class ClaimCase {}
class Report {}
class Estimator {}
class CompensationPayment {}

Contract "0..*" -- "1" Customer
Contract "1" -- "0..*" Invoice
Contract "1" -- "0..*" InsurancePolicy
Contract "1" -- "0..*" ClaimCase

ClaimCase "1" -- "0..*" CompensationPayment
ClaimCase "1" -- "0..*" Report
Report "0..*" -- "1" Estimator
BrokerCustomerAssignment "0..1" -- "1" Customer
Broker "1" -- "0..*" BrokerCustomerAssignment

@enduml

In [ ]:
for model in MODEL_MLX:
    import gc, torch
    gc.collect()
    torch.mps.empty_cache()
    print(f"Few shot with {model}")
    mlx_two(model)
    

Few shot with mlx-community/phi-4-8bit


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 48/48 [18:45<00:00, 23.45s/it]


Few shot with mlx-community/Falcon3-10B-Instruct-6bit


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

...

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 48/48 [08:32<00:00, 10.67s/it]  


Few shot with mlx-community/gemma-3-4b-it-5bit


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

100%|██████████| 48/48 [05:06<00:00,  6.38s/it]


### Clean Cache

In [ ]:
import gc, torch
gc.collect()
torch.mps.empty_cache()